# Waves2SurfNet: data loading and training, step by step

This notebook opens up the steps that are normally handled by `run_training`. It is intended for an interactive GPU job and for experimenting with data-loading choices.

We will explicitly:

1. choose spatial inputs, metadata, and velocity targets;
2. split time samples into training and validation sets;
3. construct `NetCDFFieldDataset` objects;
4. wrap them in PyTorch `DataLoader`s;
5. inspect one mini-batch;
6. build `Waves2SurfNet`;
7. perform a single optimization step; and
8. write small, readable training and validation loops.

The values below are examples. Change paths, variable names, normalization statistics, and split boundaries to match your data.

## 1. Interactive-job setup

Start an interactive GPU allocation using your cluster's scheduler, activate the environment containing this project, and launch Jupyter from the `waves2surf_net` directory. For example, the shell portion might look conceptually like:

```bash
# The exact allocation command is cluster-specific.
srun --pty --gres=gpu:1 --time=02:00:00 bash
source .venv/bin/activate
jupyter lab --no-browser
```

A GPU is useful but not required for the first few cells. Start with a small spatial subset, small batch, and perhaps one epoch while checking that shapes and masks are correct.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from netCDF4 import Dataset
from torch.utils.data import DataLoader

from ocean_velocity.data import ChannelStats, NetCDFFieldDataset
from ocean_velocity.losses import masked_l1
from ocean_velocity.metrics import RegressionMetrics
from ocean_velocity.model import Waves2SurfNet

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Describe the scientific variables

`input_variables` are two-dimensional fields given to the convolutional network. Examples include SSH, gridded wind components, bathymetry, or latitude.

`metadata_variables` are global scalar values associated with a snapshot, such as domain-mean temperature. Calendar features are also global metadata and are generated from the NetCDF time coordinate.

**Calendar embedding is optional.** Use `CALENDAR_FEATURES = []` when calendar information is unavailable or when you want an ablation without seasonal conditioning. No other code needs to change: the metadata dimension is calculated from the list lengths automatically.

`target_variables` are the fields the network learns to predict. Here they are the two surface-velocity components.

The order matters: statistics, tensors, and reported metrics follow these lists exactly.

In [ ]:
# Replace this with the NetCDF file visible from the compute node.
DATA_PATH = Path("/path/to/ocean_fields.nc")

INPUT_VARIABLES = ["ssh", "wind_u", "wind_v"]
METADATA_VARIABLES = ["background_temperature"]
# Optional. Set to [] to train without calendar-date conditioning.
CALENDAR_FEATURES = ["year_sin", "year_cos"]
TARGET_VARIABLES = ["surface_u", "surface_v"]

TIME_VARIABLE = "time"
MASK_VARIABLE = "ocean_mask"  # Set to None when there is no explicit mask.

# Optional [y_start, y_stop, x_start, x_stop]. A small box is helpful while debugging.
SPATIAL_SLICE = [0, 128, 0, 128]

print("Spatial input channels:", len(INPUT_VARIABLES))
print("Metadata values:", len(METADATA_VARIABLES) + len(CALENDAR_FEATURES))
print("Target channels:", len(TARGET_VARIABLES))

### Inspect the file before constructing a dataset

This short inspection often catches incorrect variable names or unexpected dimension order immediately. The baseline reader expects time-varying spatial fields to use `[time, y, x]`, static fields to use `[y, x]`, and scalar metadata to be scalar or `[time]`.

In [ ]:
# Uncomment after setting DATA_PATH.
# with Dataset(DATA_PATH) as nc:
#     print("Dimensions:", {name: len(dim) for name, dim in nc.dimensions.items()})
#     for name in INPUT_VARIABLES + METADATA_VARIABLES + TARGET_VARIABLES:
#         variable = nc.variables[name]
#         print(f"{name:28s} shape={variable.shape}, dimensions={variable.dimensions}")
#     n_times = len(nc.dimensions[TIME_VARIABLE])

# Temporary placeholder so the splitting examples can be read before a file is configured.
n_times = 1000
print("Number of available snapshots:", n_times)

## 3A. Split one file by time index

For geophysical time series, a chronological split is usually safer than randomly assigning individual snapshots. Neighboring times can be highly correlated; putting adjacent snapshots in both training and validation can make validation performance look unrealistically good.

Below, the earliest 70% is training data, the next 15% is validation data, and the final 15% is held out for testing. The test indices are defined but are not used while choosing a model.

For strongly autocorrelated data, consider leaving a temporal gap between splits.

In [ ]:
train_stop = int(0.70 * n_times)
validation_stop = int(0.85 * n_times)

train_indices = np.arange(0, train_stop)
validation_indices = np.arange(train_stop, validation_stop)
test_indices = np.arange(validation_stop, n_times)

print("Train snapshots:", len(train_indices))
print("Validation snapshots:", len(validation_indices))
print("Test snapshots:", len(test_indices))

### Normalization

Variables with different units and magnitudes should be standardized before training. Means and standard deviations must be calculated from the training indices only; using validation/test values leaks information.

The numbers below are placeholders. Use `calculate_statistics.py` for a real experiment, then copy its output here or into the JSON configuration. The statistics follow the variable-list order.

In [ ]:
input_stats = ChannelStats(
    mean=(0.0, 0.0, 0.0),
    std=(0.2, 5.0, 5.0),
)
metadata_stats = ChannelStats(
    # background_temperature, year_sin, year_cos
    mean=(15.0, 0.0, 0.0),
    std=(5.0, 0.7071, 0.7071),
)
target_stats = ChannelStats(
    mean=(0.0, 0.0),
    std=(0.5, 0.5),
)

### Construct the datasets directly

A PyTorch `Dataset` describes how to obtain one sample. It does not decide batching or shuffling. Both datasets below use the same variables and normalization but different time indices.

In [ ]:
common_dataset_options = dict(
    input_variables=INPUT_VARIABLES,
    target_variables=TARGET_VARIABLES,
    metadata_variables=METADATA_VARIABLES,
    time_variable=TIME_VARIABLE,
    calendar_features=CALENDAR_FEATURES,
    valid_mask_variable=MASK_VARIABLE,
    spatial_slice=SPATIAL_SLICE,
    input_stats=input_stats,
    metadata_stats=metadata_stats,
    target_stats=target_stats,
)

# Uncomment after setting a real DATA_PATH.
# train_dataset = NetCDFFieldDataset(
#     DATA_PATH, indices=train_indices, **common_dataset_options
# )
# validation_dataset = NetCDFFieldDataset(
#     DATA_PATH, indices=validation_indices, **common_dataset_options
# )
# print(len(train_dataset), len(validation_dataset))

## 3B. Alternative: use pre-partitioned files

Some groups prepare separate training and validation NetCDF files. This is useful when the files represent different years, regions, simulations, or observing platforms.

Construct one dataset per file and give each all of its own time indices. Normalization must still come only from the training file. This strategy does not require changing the model or DataLoader.

In [ ]:
TRAIN_PATH = Path("/path/to/train_fields.nc")
VALIDATION_PATH = Path("/path/to/validation_fields.nc")

# with Dataset(TRAIN_PATH) as nc:
#     train_file_indices = np.arange(len(nc.dimensions[TIME_VARIABLE]))
# with Dataset(VALIDATION_PATH) as nc:
#     validation_file_indices = np.arange(len(nc.dimensions[TIME_VARIABLE]))
#
# train_dataset = NetCDFFieldDataset(
#     TRAIN_PATH, indices=train_file_indices, **common_dataset_options
# )
# validation_dataset = NetCDFFieldDataset(
#     VALIDATION_PATH, indices=validation_file_indices, **common_dataset_options
# )

## 4. Wrap datasets in DataLoaders

A `DataLoader` groups samples into mini-batches and optionally reads them in worker processes.

- Shuffle training data so consecutive optimizer steps do not repeatedly see one season or regime.
- Do not shuffle validation data; deterministic order makes debugging easier.
- Batch size trades memory for throughput. Begin with 2–8 for large fields and increase while monitoring GPU memory.
- `num_workers` controls parallel CPU readers, not GPUs. Start at 0 while debugging, then try 2–8.
- The dataset opens a separate NetCDF handle lazily in each worker.

In [ ]:
BATCH_SIZE = 4
NUM_WORKERS = 0  # Increase after the basic pipeline works.

loader_options = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

# train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
# validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)

## 5. Inspect one batch before training

Never skip this step when adapting the data pipeline. Confirm channel order, tensor shapes, finite values, mask coverage, and approximate normalized ranges.

DataLoader adds a leading batch dimension:

- `x`: `[batch, spatial_input_channels, height, width]`
- `metadata`: `[batch, metadata_values]`
- `y`: `[batch, 2, height, width]`
- `valid_mask`: `[batch, 1, height, width]`

In [ ]:
# batch = next(iter(train_loader))
# for name, value in batch.items():
#     print(f"{name:15s} shape={tuple(value.shape)}, dtype={value.dtype}")
#
# print("Input mean/std:", batch["x"].mean().item(), batch["x"].std().item())
# print("Valid ocean fraction:", batch["valid_mask"].float().mean().item())
# print("Original time indices:", batch["index"].tolist())

## 6. Construct Waves2SurfNet

This example uses the four-channel metadata broadcast method. The metadata MLP compresses the global metadata vector to four learned values, broadcasts them across the grid, and concatenates them with the spatial fields.

Other controlled experiments need only change `conditioning`:

- `none`: ignore metadata;
- `extra_channels`: broadcast every metadata value directly;
- `broadcast`: MLP-compress metadata to four channels; or
- `film`: modulate features throughout the network.

GroupNorm is a good default for the small batches common with large fields.

In [ ]:
# This remains correct when CALENDAR_FEATURES is empty.
metadata_dim = len(METADATA_VARIABLES) + len(CALENDAR_FEATURES)

model = Waves2SurfNet(
    in_channels=len(INPUT_VARIABLES),
    out_channels=len(TARGET_VARIABLES),
    base_channels=8,
    normalization="group",
    metadata_dim=metadata_dim,
    conditioning="broadcast",
    metadata_channels=4,
    metadata_width=32,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Device:", device)
print(f"Trainable parameters: {trainable_parameters:,}")

## 7. Perform one optimization step explicitly

A training step has five essential operations:

1. move one batch to the GPU;
2. ask the model for a prediction;
3. calculate prediction error;
4. backpropagate gradients; and
5. let the optimizer update the model parameters.

The valid mask prevents land and missing observations from contributing to loss.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

# batch = next(iter(train_loader))
# x = batch["x"].to(device)
# metadata = batch["metadata"].to(device)
# target = batch["y"].to(device)
# mask = batch["valid_mask"].to(device)
#
# model.train()
# optimizer.zero_grad(set_to_none=True)
# prediction = model(x, metadata)
# loss = masked_l1(prediction, target, mask)
# loss.backward()
# optimizer.step()
#
# print("Prediction shape:", prediction.shape)
# print("One-batch masked L1 loss:", loss.item())

## 8. Define readable epoch functions

The functions below contain the same core operations as the single-step example. A training epoch updates model weights; validation uses `model.eval()` and `torch.no_grad()` so it measures performance without changing the network.

Metrics are accumulated over all valid pixels. Because targets are standardized during training, the validation function converts predictions and targets back to physical units before reporting RMSE, MAE, and R².

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """Update model parameters using every mini-batch once."""
    model.train()
    total_loss = 0.0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        metadata = batch["metadata"].to(device, non_blocking=True)
        target = batch["y"].to(device, non_blocking=True)
        mask = batch["valid_mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        prediction = model(x, metadata)
        loss = masked_l1(prediction, target, mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


def validate(model, loader, device, target_stats):
    """Measure loss and physical-unit metrics without changing weights."""
    model.eval()
    metrics = RegressionMetrics(channels=len(TARGET_VARIABLES))
    total_loss = 0.0

    mean = torch.tensor(target_stats.mean, device=device)[None, :, None, None]
    std = torch.tensor(target_stats.std, device=device)[None, :, None, None]

    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device, non_blocking=True)
            metadata = batch["metadata"].to(device, non_blocking=True)
            target = batch["y"].to(device, non_blocking=True)
            mask = batch["valid_mask"].to(device, non_blocking=True)

            prediction = model(x, metadata)
            total_loss += masked_l1(prediction, target, mask).item()

            prediction_physical = prediction * std + mean
            target_physical = target * std + mean
            metrics.update(prediction_physical, target_physical, mask)

    result = metrics.compute()
    result["normalized_l1"] = total_loss / max(len(loader), 1)
    return result

## 9. Run a short experiment

For an initial interactive check, run one or two epochs. A scientifically meaningful model will usually require more data and epochs, but this short run verifies that data loading, GPU transfer, loss, gradients, and validation all work together.

The best checkpoint should be selected using validation loss. Do not repeatedly inspect test performance while designing the model.

In [ ]:
N_EPOCHS = 2
best_validation_loss = float("inf")

# for epoch in range(1, N_EPOCHS + 1):
#     train_loss = train_one_epoch(model, train_loader, optimizer, device)
#     validation = validate(model, validation_loader, device, target_stats)
#
#     print(
#         f"Epoch {epoch:02d} | train L1={train_loss:.4f} | "
#         f"validation L1={validation['normalized_l1']:.4f} | "
#         f"RMSE={validation['rmse']}"
#     )
#
#     if validation["normalized_l1"] < best_validation_loss:
#         best_validation_loss = validation["normalized_l1"]
#         torch.save(
#             {"model": model.state_dict(), "epoch": epoch},
#             "tutorial_best.pt",
#         )

## 10. Productive data-loading experiments

Once the basic notebook runs, change one decision at a time:

- Compare a chronological split with held-out years or regions.
- Vary `SPATIAL_SLICE` to test small patches before full domains.
- Increase `NUM_WORKERS` and time one epoch to tune I/O throughput.
- Increase batch size until GPU memory is nearly full, leaving some margin.
- Compare `conditioning='none'`, `'extra_channels'`, `'broadcast'`, and `'film'` using identical splits and seeds.
- Add or remove physical fields while keeping channel order and statistics aligned.
- Plot masks, inputs, targets, and predictions before trusting summary metrics.

If random spatial crops or flips are added later, remember that vector components must transform consistently. A horizontal reflection, for example, changes the sign of the corresponding vector component; treating vector fields as ordinary images can create physically incorrect examples.

## Relation to the command-line pipeline

This notebook constructs each object explicitly for learning and experimentation. Production runs should generally return to the configuration-driven commands:

```bash
python calculate_statistics.py --config configs/example.json --output normalization.json
python train.py --config configs/example.json --output-dir runs/baseline
python evaluate.py --checkpoint runs/baseline/best.pt --split test
```

Those scripts call the same dataset, model, losses, and metrics demonstrated here, while also saving configuration, history, and complete checkpoints reproducibly.